# EDA: Housing Price

In [ ]:
import os
import pandas as pd

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Load Dataset

In [ ]:
path = os.path.join(os.getcwd(), "..", "..", "datasets", "housing.csv")
df = pd.read_csv(path, sep=",")
df.shape

# Initial Exploration

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

### Missing Values

In [ ]:
df.isna().sum()

### Duplicates

In [ ]:
df.duplicated().sum()

# Univariate Analysis

## Categorical Features

In [ ]:
df["ocean_proximity"].value_counts()

In [ ]:
cat_feature = "ocean_proximity"
# plot histogram of cat_feature
plt.figure(figsize=(8, 5))
sns.countplot(x=cat_feature, data=df, palette="Set2")
plt.title(f"Distribution of {cat_feature}")
plt.xlabel(cat_feature)
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

## Numerical Features

In [ ]:
num_features = df.select_dtypes(include=["float64"]).columns.tolist()
len(num_features)

In [ ]:
# numerical features
num_features = df.select_dtypes(include=["float64"]).columns.tolist()
n = len(num_features)
ncols = 4
nrows = -(-n // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df[col], bins=50, color="steelblue", edgecolor="none", alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    skew = df[col].skew()
    axes[i].text(
        0.97,
        0.95,
        f"skew={skew:.2f}",
        transform=axes[i].transAxes,
        ha="right",
        va="top",
        fontsize=9,
        color="gray",
    )

plt.suptitle("Feature Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots of numerical features
n = len(num_features)
ncols = 4
nrows = -(-n // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].boxplot(df[col])
    axes[i].set_title(col)

plt.suptitle("Boxplots of Numerical Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Bivariate Analysis

### Correlation Analysis

In [ ]:
corr = df[num_features].corr()
TARGET = "median_house_value"
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Full heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=axes[0],
    linewidths=0.4,
)
axes[0].set_title("Correlation matrix (lower triangle)")

# Correlation with target only
target_corr = corr[TARGET].drop(TARGET).sort_values()
colors = ["salmon" if v < 0 else "steelblue" for v in target_corr]
target_corr.plot(kind="barh", ax=axes[1], color=colors, edgecolor="none")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Correlation with {TARGET}")
axes[1].set_xlabel("Pearson r")

plt.tight_layout()
plt.show()

In [ ]:
# plot pairplots
sample_pp = df[num_features].sample(1500, random_state=42)

g = sns.pairplot(
    sample_pp,
    diag_kind="kde",
    plot_kws={"alpha": 0.3, "s": 10},
    diag_kws={"fill": True},
)
g.figure.suptitle("Pairplot", y=1.01)
plt.show()

## **Key Observations:**
- the shape of dataset: (20640,10)
- `total_bedrooms` has 207 missed value.
- no duplicated values

**Categorical features:**

- `ocean_proximity` has few values of category ISLAND.

**Numerical Features:**

- `population`, `total_bedrooms`, `total_rooms`, `households` are highly right skewed.
-  those features give us informations per `households` and not per `house`.
-  there is negative correlation between target and features: `longitude`,`latitude` and `population`

**Correlated Features:**
- `households` and `total_bedrooms` highly correlated,
- `households` and `total_rooms` highly correlated,
- `total_bedrooms` and `total_rooms` highly correlated,

### **Decisions to be made:**

- handle missing values of `total_bedrooms`.

**Categorical Features:**

- remove the value ISLAND of the feature `ocean_proximity`
- Encode the feature `ocean_proximity`

**Numerical Features:**

- handle the skeweness of `population`, `total_bedrooms`, `total_rooms`, `households`
- create new features by processing the original ones, so they will turn out to be more predictive.



